# Phase 1: Scalarized Bayesian Optimization for 200 MeV Electron Injector Linac

This interactive notebook executes **Phase 1 Scalarized Bayesian Optimization (BO)** for the 200 MeV electron injector linac simulation. Scalarized BO combines multiple beam quality objectives into a single scalar merit function using weight combinations:

$$f(\mathbf{x}) = w_1 \cdot \varepsilon_{n,x} + w_2 \cdot \varepsilon_{n,y} + w_3 \cdot \sigma_E$$

A single Gaussian Process surrogate (`SingleTaskGP`) with Matérn 5/2 ARD kernel is fitted to the scalarized objective, and candidates are selected using `qLogNoisyExpectedImprovement` (`qLogNEI`).

In [ ]:
# Useful for interactive debugging and live code updates
%load_ext autoreload
%autoreload 2

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Ensure project root is in path
import sys
sys.path.insert(0, str(Path("..").resolve()))

from mobo_linac.config import load_config
from mobo_linac.execution.parallel import BatchEvaluator
from mobo_linac.cli import run_scalarized

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [ ]:
# ── Simulation Configuration Summary ─────────────────────────────────────────
# Displays all key parameters for this notebook run in a tabular format.
# Defaults are values defined in configs/mobo_200MeV.yaml.
# Reconfigured values are overrides applied below for this interactive session.

import pandas as pd

# --------------------------------------------------------------------------
# Notebook-level overrides (edit these to reconfigure a run)
# --------------------------------------------------------------------------
_WEIGHTS       = [1.0, 1.0, 1.0]   # default: equal weighting
_N_ITERATIONS  = 10                 # default (script): 300 | reduced for interactive use
_BATCH_SIZE    = 4                  # default (script):   8 | reduced for interactive use
_INIT_SAMPLES  = 8                  # default (script):  16 | reduced for interactive use
_NUM_WORKERS   = 4                  # default (script):   4
_SEED          = 42                 # default: 42
_CONFIG_FILE   = "configs/mobo_200MeV.yaml"  # default config

# --------------------------------------------------------------------------
# Build summary table
# --------------------------------------------------------------------------
_rows = [
    # (Parameter, Default, This Session, Unit/Notes)
    ("Config file",        "configs/mobo_200MeV.yaml", _CONFIG_FILE,              "YAML"),
    ("Objective weights",  "[1.0, 1.0, 1.0]",          str(_WEIGHTS),             "[w_ex, w_ey, w_sE]"),
    ("BO iterations",      "300",                       str(_N_ITERATIONS),        "script default=300"),
    ("Batch size (q)",     "8",                         str(_BATCH_SIZE),          "candidates/iter"),
    ("Initial samples",   "16",                         str(_INIT_SAMPLES),        "Sobol random init"),
    ("Parallel workers",  "4",                          str(_NUM_WORKERS),         "ASTRA processes"),
    ("Random seed",       "42",                         str(_SEED),                "reproducibility"),
    ("Surrogate model",   "SingleTaskGP",               "SingleTaskGP",            "Matérn-5/2 ARD kernel"),
    ("Acquisition fn",    "qLogNEI",                    "qLogNEI",                 "log Noisy EI"),
    ("Objectives",        "ex, ey, sigma_E",            "ex, ey, sigma_E",         "minimise all 3"),
    ("Design variables",  "6D",                         "6D",                      "sol, Q1, Q2, phi_gun, phi12, phi34"),
    ("ASTRA binary",      "./bin/astra",                "./bin/astra",             "$PROJECT_ROOT/bin"),
]

_df = pd.DataFrame(_rows, columns=["Parameter", "Default", "This Session", "Notes"])
_df["Changed"] = _df.apply(
    lambda r: "✔ reconfigured" if r["Default"] != r["This Session"] else "", axis=1
)

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║   Phase 1 · Scalarized BO · Simulation Configuration Summary         ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
display(_df.to_string(index=False))
try:
    from IPython.display import display as _disp
    _disp(_df.style
         .set_caption("Phase 1 Scalarized BO — Configuration Summary")
         .applymap(lambda v: "background-color:#ffeeba;font-weight:bold" if v == "✔ reconfigured" else "",
                   subset=["Changed"])
         .set_properties(**{"text-align": "left"})
         .hide(axis="index")
    )
except Exception:
    pass


## 1. Environment & Binary Setup

Verify environment variable settings for local ASTRA binaries:

In [ ]:
project_root = Path("..").resolve()
bin_dir = project_root / "bin"
astra_bin = bin_dir / "astra"
generator_bin = bin_dir / "generator"

os.environ["ASTRA_BIN"] = str(astra_bin)
os.environ["GENERATOR_BIN"] = str(generator_bin)

print(f"ASTRA_BIN: {os.environ.get('ASTRA_BIN')}")
print(f"GENERATOR_BIN: {os.environ.get('GENERATOR_BIN')}")
print(f"Binary exists: {astra_bin.exists()}")

## 2. Load Configuration

Load central YAML configuration parameters (6D decision variables, beam constraints, reference points):

In [ ]:
config_path = project_root / "configs" / "mobo_200MeV.yaml"
config = load_config(config_path)
print(f"Loaded config: {config.name} (version {config.version})")
print(f"Design Variables: {len(config.design_variables)}")
for p in config.design_variables:
    print(f"  - {p.name} [{p.astra_name}]: bounds = [{p.bounds[0]}, {p.bounds[1]}] {p.unit}")

## 2.5 Surrogate Hyperparameter Optimization & Model Selection

Perform systematic cross-validation over kernel types (`matern52`, `rbf`) and observation noise ratios to identify the optimal GP surrogate configuration for the linac objectives:

In [ ]:
from mobo_linac.models.tuning import tune_gp_hyperparameters

# Sample initial Sobol evaluation points for surrogate diagnostic tuning
bounds_tensor = config.get_parameter_bounds_tensor()
sobol_eng = torch.quasirandom.SobolEngine(dimension=bounds_tensor.shape[1], scramble=True, seed=42)
init_X = bounds_tensor[0] + (bounds_tensor[1] - bounds_tensor[0]) * sobol_eng.draw(16).to(dtype=torch.double)

# Synthetic multi-scale objective values for offline surrogate tuning demonstration
init_Y = torch.cat([
    1.0e-6 * (1.0 + 0.1 * torch.sin(init_X[:, 0:1])),
    1.0e-6 * (1.0 + 0.1 * torch.cos(init_X[:, 1:2])),
    1.0e6  * (1.0 + 0.1 * torch.sin(init_X[:, 2:3])),
], dim=-1)

tuning_summary = tune_gp_hyperparameters(
    train_X=init_X,
    train_Y=init_Y,
    bounds=bounds_tensor,
    candidate_covars=["matern52", "rbf"],
    candidate_noise_ratios=[1.0e-8, 1.0e-6, 1.0e-4],
    candidate_noise_modes=["deterministic_fixed"],
    objective_names=["norm_emit_x", "norm_emit_y", "sigma_energy"],
)

print("=== Surrogate Hyperparameter Optimization Results ===")
print(f"Optimal Kernel:      {tuning_summary.best_config.covar_type}")
print(f"Optimal Noise Mode:  {tuning_summary.best_config.noise_mode}")
print(f"Optimal Noise Ratio: {tuning_summary.best_config.relative_noise_ratio:.1e}")
print(f"Overall R^2 Score:   {tuning_summary.best_candidate.overall_r2:.4f}")
display(tuning_summary.comparison_table)

## 3. Configure and Execute Scalarized BO Campaign

Execute Phase 1 scalarized BO with custom weight vectors (e.g. equal weights `[1.0, 1.0, 1.0]`):

In [ ]:
class ScalarizedBOArgs:
    config = str(config_path)
    n_iterations = 20
    batch_size = 8
    num_initial_samples = 16
    num_workers = 4
    weights = [1.0, 1.0, 1.0]
    seed = 42
    output_dir = str(project_root / "results_notebooks" / "phase1_scalarized")

args = ScalarizedBOArgs()
print(f"Running Phase 1 Scalarized BO to {args.output_dir}...")
# Uncomment to run simulation:
# run_scalarized(args)

## 4. Analyze Results & Trade-off Plots

Load results CSVs and plot objective progression:

In [ ]:
results_dir = project_root / "results_notebooks" / "phase1_scalarized"
if (results_dir / "train_Y.csv").exists():
    df_y = pd.read_csv(results_dir / "train_Y.csv")
    print("Loaded evaluations:", len(df_y))
    print(df_y.head())
    
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(df_y['emittance_x'], 'b-o', label='\epsilon_{n,x}')
    ax[0].set_title('Horizontal Emittance [mm mrad]')
    ax[0].grid(True)
    
    ax[1].plot(df_y['emittance_y'], 'g-o', label='\epsilon_{n,y}')
    ax[1].set_title('Vertical Emittance [mm mrad]')
    ax[1].grid(True)
    
    ax[2].plot(df_y['energy_spread'], 'r-o', label='\sigma_E')
    ax[2].set_title('RMS Energy Spread [MeV]')
    ax[2].grid(True)
    plt.tight_layout()
    plt.show()
else:
    print(f"No run output found at {results_dir}. Execute the run cell above first.")

## 5. Extended Visualizations

Rich plotting suite for Phase 1 Scalarized BO analysis. These cells load results from `results_dir` and call the `mobo_linac.plotting` module functions.

> **Note**: Run the BO campaign in Cell 9 (or point `results_dir` at an existing run) before executing these cells.

In [ ]:
# ── 5.1 Import plotting module ───────────────────────────────────────────────
from mobo_linac.plotting import (
    plot_hypervolume_progress,
    plot_pareto_front,
    plot_pareto_front_3d,
    plot_objective_evolution,
    plot_best_so_far,
    plot_feasibility_rate,
    plot_constraint_diagnostics,
    plot_constraint_violins,
    plot_design_variable_heatmap,
    plot_parallel_coordinates,
    plot_scalarized_objective_trace,
)
from mobo_linac.io.results import load_results

# Load results (requires a completed or resumed run)
results_dir = project_root / "results_notebooks" / "phase1_scalarized"
figures_dir = results_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# Load EvaluationResult objects if available
try:
    results = load_results(results_dir)
    print(f"Loaded {len(results)} evaluation results.")
except Exception as e:
    results = []
    print(f"No results found: {e}  — generate results first by running Cell 9.")


### 5.2 Hypervolume Progress

In [ ]:
# Hypervolume progress from hypervolume.csv
hv_csv = results_dir / "hypervolume.csv"
if hv_csv.exists():
    hv_df = pd.read_csv(hv_csv)
    fig = plot_hypervolume_progress(hv_df,
                                   output_path=figures_dir / "hypervolume_progress.png")
    plt.show()
else:
    print("hypervolume.csv not found — run BO campaign first.")


### 5.3 Scalarized Objective Trace  *(Phase 1 specific)*

In [ ]:
# Scalarized merit function f(x) = w1·ε_x + w2·ε_y + w3·σ_E over evaluations
if results:
    fig = plot_scalarized_objective_trace(
        results,
        weights=_WEIGHTS,   # from Configuration Summary cell
        output_path=figures_dir / "scalarized_trace.png",
    )
    plt.show()


### 5.4 Objective Evolution & Best-So-Far

In [ ]:
# Individual objective time-series with running minimum
if results:
    fig = plot_objective_evolution(results,
                                  output_path=figures_dir / "objective_evolution.png")
    plt.show()

    fig = plot_best_so_far(results,
                          output_path=figures_dir / "best_so_far.png")
    plt.show()


### 5.5 Pareto Front Projections

In [ ]:
# 2D Pareto projections (3 panels)
if results:
    fig = plot_pareto_front(results,
                           output_path=figures_dir / "pareto_2d.png")
    plt.show()

# 3D Pareto scatter (interactive in Jupyter; rotate with mouse)
if results:
    fig = plot_pareto_front_3d(results, elev=25, azim=45,
                               output_path=figures_dir / "pareto_3d.png")
    plt.show()


### 5.6 Feasibility Rate

In [ ]:
# Cumulative and rolling feasibility rate over BO campaign
if results:
    fig = plot_feasibility_rate(results, window=10,
                               output_path=figures_dir / "feasibility_rate.png")
    plt.show()


### 5.7 Constraint Diagnostics & Violin Plots

In [ ]:
# Time-series of all beam-quality constraint diagnostics
if results:
    fig = plot_constraint_diagnostics(results,
                                     output_path=figures_dir / "constraint_diagnostics.png")
    plt.show()

# Violin distributions — feasible vs infeasible
if results:
    fig = plot_constraint_violins(results,
                                 output_path=figures_dir / "constraint_violins.png")
    plt.show()


### 5.8 Design Variable Analysis

In [ ]:
# Pearson correlation heatmap between 6 design variables (feasible only)
if results:
    fig = plot_design_variable_heatmap(results, feasible_only=True,
                                      output_path=figures_dir / "design_var_heatmap.png")
    plt.show()

# Parallel coordinates — design space coloured by ε_nx
if results:
    fig = plot_parallel_coordinates(
        results,
        color_by="norm_emit_x_m_rad",
        feasible_only=True,
        n_lines=200,
        output_path=figures_dir / "parallel_coordinates.png",
    )
    plt.show()


### 5.9 GP Surrogate Posterior Slice

Visualise the learned GP surrogate mean and uncertainty along one design variable axis.

In [ ]:
# GP surrogate slice — requires a fitted model object
# Replace `fitted_model` and `train_X_tensor` with the actual model from your BO loop.
# Example (uncomment after fitting a model):
#
# import torch
# from mobo_linac.plotting import plot_gp_surrogate_slice
# from mobo_linac.config import load_config
#
# cfg = load_config(str(project_root / 'configs' / 'mobo_200MeV.yaml'))
# bounds = torch.tensor([[v.lower for v in cfg.design_variables],
#                        [v.upper for v in cfg.design_variables]], dtype=torch.double)
# fixed_x = bounds.mean(dim=0)  # midpoint as reference
#
# for dim in range(6):
#     fig = plot_gp_surrogate_slice(
#         fitted_model, bounds, fixed_x,
#         dim=dim, obj_idx=0,
#         output_path=figures_dir / f'gp_slice_dim{dim}.png',
#     )
#     plt.show()
print('GP surrogate slice: uncomment the block above and supply a fitted model.')
